# Save final adata object

**Pinned Environment:** [`conda_envs/space2_20250604.yml`](../conda_envs/space2_20250604.yml)  

In [1]:
from pathlib import Path
import sys
import os
import time
import warnings
import re
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata
import scanpy as sc
import squidpy as sq


import scipy
from scipy.spatial import distance_matrix
from scipy.spatial.distance import cdist
from scipy import stats
import scipy.ndimage as ndi
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks

from shapely.geometry import Point

from shapely.geometry import MultiPolygon
from alphashape import alphashape
from alphashape import optimizealpha
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer

import pickle
import joblib

import random
import matplotlib.colors as mcolors

import matplotlib as mpl
mpl.rcParams['axes.titlesize'] = 24
mpl.rcParams['pdf.fonttype'] = 42

import warnings
warnings.simplefilter(action='ignore', category=Warning)

sys.version_info

sys.version_info(major=3, minor=12, micro=2, releaselevel='final', serial=0)

## Local file info

In [2]:
sys.path.append(str(Path.cwd().resolve().parents[0]))

from config.paths import BASE_OUTDIR, INPUTS_DIR, FUNCTIONS_DIR

inputs_dir = INPUTS_DIR
dist_out_dir = os.path.join(BASE_OUTDIR, "downstream_analysis/distance")


plot_out_dir = os.path.join(dist_out_dir, 'plots')

print(inputs_dir)
print(dist_out_dir)

/home/workspace/spatial_tls_manuscript/inputs
/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance


## Load adata

In [8]:
adata = sc.read_h5ad(os.path.join(dist_out_dir,'adata_distance_zones_structure_tls_dist_categories.h5ad'))

## Save final processed adata object for manuscript

In [58]:
# Remove adata.obs columns with only nan values - these are carried over from merging with the single cell reference
nan_cols = adata.obs.columns[adata.obs.isna().all()]
print(nan_cols.tolist())
adata.obs = adata.obs.loc[:, ~adata.obs.isna().all()]

['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'Barcode', 'Age', 'Oxygen', 'percent.mt', 'S.Score', 'G2M.Score', 'Phase', 'Sample', 'RNA_snn_res.0.2', 'seurat_clusters', 'RNA_snn_res.0.15', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.2', 'integrated_snn_res.0.1', 'cluster_high_res', 'CellType', 'leiden_res1', 'CellType_consolidated']


In [60]:
dist_out_dir

'/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance'

In [62]:
adata.write_h5ad(os.path.join(dist_out_dir,'adata_HDM_processed.h5ad'), compression='gzip')

In [4]:
adata = sc.read_h5ad(os.path.join(dist_out_dir,'adata_HDM_processed.h5ad'))

# Upload to HISE

### The following were run in terminal:
gcloud auth login 
gsutil -m cp  /home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance/adata_HDM_processed.h5ad gs://wftissueimmaifi-219b3400-2a33-483a-a457-d5272e41d7db